In [ ]:
import ast
import pandas as pd
from datasets import Dataset
from openai import OpenAI

from ragas import evaluate
from ragas.llms import llm_factory
from ragas.embeddings import HuggingFaceEmbeddings
from ragas.metrics import answer_correctness

# Load full dataset
df = pd.read_csv("ragas_dataset.csv")
df["contexts"] = df["contexts"].apply(ast.literal_eval)

# Legacy evaluate() expects ground_truth
legacy_df = df.rename(columns={"question": "user_input", "answer": "response", "reference": "reference", "contexts": "retrieved_contexts"})

dataset = Dataset.from_pandas(
    legacy_df[["implementation", "scenario_id", "user_input", "response", "reference", "retrieved_contexts"]],
    preserve_index=False
)

# Local Ollama evaluator
client = OpenAI(
    api_key="ollama",
    base_url="http://localhost:11434/v1",
)

evaluator_llm = llm_factory(
    "llama3.1",
    provider="openai",
    client=client,
    max_tokens=4096,
    temperature=0.0,
    top_p=1.0,
)

# Local embeddings
embeddings = HuggingFaceEmbeddings(
    model="sentence-transformers/all-MiniLM-L6-v2"
)

# Evaluate answer correctness only
result = evaluate(
    dataset=dataset,
    metrics=[answer_correctness],
    llm=evaluator_llm,
    embeddings=embeddings,
    show_progress=False,
)

df_scores = result.to_pandas()
df_eval_answer = pd.concat([df[["implementation", "scenario_id"]], df_scores], axis=1)

df_eval_answer.to_csv("ragas_answer_correctness.csv", index=False)
df_eval_answer

/Users/nikos/Thesis/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/var/folders/0k/m_yklbmx2fqb7t9rj_3gv4q80000gn/T/ipykernel_26824/3081769309.py:9: DeprecationWarning: Importing answer_correctness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_correctness
  from ragas.metrics import answer_correctness
Loading weights: 100%|██████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 9469.62it/s]
Exception raised in Job[0]: TimeoutError()
Exception raised in Job[1]: TimeoutError()
Exception raised in Job[2]: TimeoutError()
Exception raised in Job[3]: TimeoutError()
Exception raised in Job[4]: TimeoutError()
Exception raised in Job[5]: TimeoutError()
Excep

In [ ]:
summary_answer = (
    df_eval_answer.groupby("implementation")[["answer_correctness"]]
    .mean()
    .sort_values("answer_correctness", ascending=False)
)

summary_answer

In [ ]:
scenario_answer = (
    df_eval_answer.groupby(["implementation", "scenario_id"])[["answer_correctness"]]
    .mean()
    .reset_index()
)

scenario_answer